In [16]:
import pandas as pd
import numpy as np

In [12]:
traffic_df = pd.read_csv('data/Traffic_Crashes.csv')
red_light_df = pd.read_csv('data/Red_Light.csv')
#clean all dataframes by long and lattitude
upper_left_lat = 41.954500 #W Irving Park Rd
upper_left_lon = -87.745480 #Cicero Ave
lower_right_lat = 41.765300 #71st St
lower_right_lon = -87.550018 #Lake Michigan


def clean_dataframe(df):
    df = df.dropna(subset=['LATITUDE', 'LONGITUDE'])
    df = df[(df['LATITUDE'] >= lower_right_lat) & (df['LATITUDE'] <= upper_left_lat) &
            (df['LONGITUDE'] >= upper_left_lon) & (df['LONGITUDE'] <= lower_right_lon)]
    return df

#drop rows with missing values in INJURIES_TOTAL and filter for relevant traffic control devices that indicate intersections
traffic_df = clean_dataframe(traffic_df)
traffic_df = traffic_df.dropna(subset=['INJURIES_TOTAL'])
traffic_df = traffic_df[traffic_df['TRAFFIC_CONTROL_DEVICE'].isin(['FLASHING CONTROL SIGNAL', 'TRAFFIC SIGNAL'])]

red_light_df = clean_dataframe(red_light_df)

C:\Users\jessi\AppData\Local\Temp\ipykernel_25404\809174387.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  traffic_df = pd.read_csv('data/Traffic_Crashes.csv')


In [13]:
#write cleaned dataframes to new csv files
traffic_df.to_csv('data/cleaned_Traffic_Crashes.csv', index=False)
red_light_df.to_csv('data/cleaned_Red_Light.csv', index=False)

In [ ]:
from sklearn.neighbors import BallTree

red_light_df = red_light_df.reset_index(drop=True)
red_light_df['red_light_id'] = np.arange(1, len(red_light_df) + 1)

# convert to radians for haversine distance
traffic_rad = np.radians(traffic_df[['LATITUDE', 'LONGITUDE']].values)
red_rad = np.radians(red_light_df[['LATITUDE', 'LONGITUDE']].values)

# build spatial index on red light cameras
tree = BallTree(red_rad, metric='haversine')

# find nearest red light camera for each crash
dist, ind = tree.query(traffic_rad, k=1)
traffic_df['dist_to_red_light_m'] = dist[:, 0] * 6371000  # Earth radius in meters

# threshold = 50 meters for "near" a red light camera
threshold = 50

# create binary indicator
traffic_df['has_red_light'] = (traffic_df['dist_to_red_light_m'] <= threshold).astype(int)
traffic_df['matched_red_light_id'] = red_light_df.iloc[ind[:, 0]]['red_light_id'].values


traffic_matched_df = traffic_df.copy()
traffic_matched_df.to_csv('data/crashes_camera.csv', index=False)

In [15]:
traffic_df.head(5)

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,dist_to_red_light_m,has_red_light,matched_red_light_id
11,05a1ef7524fa337f7bc45d41817be4bdbba7778ea24851...,NaN,04/05/2026 09:40:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,"DARKNESS, LIGHTED ROAD",REAR END,FOUR WAY,...,0.0,21,1,4,41.779949,-87.625290,POINT (-87.625289553338 41.779949104569),23.308203,1,37
23,b9d5dbd867a7bb0eb762aafc7e9e6f97e7c86f0dbe1f98...,Y,04/05/2026 07:28:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DARKNESS,REAR END,NOT DIVIDED,...,0.0,19,1,4,41.809237,-87.631867,POINT (-87.631867098157 41.809237163628),1649.588022,0,39
29,88fadca28781059ae241289f924d438a5c7a40eaded94c...,NaN,04/05/2026 07:10:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DUSK,PEDESTRIAN,NOT DIVIDED,...,0.0,19,1,4,41.864681,-87.646990,POINT (-87.646990408329 41.864681333354),263.247700,0,162
30,4c7e39bbb7d7ded2a5c35b131018dc3a21a47e9fb761aa...,NaN,04/05/2026 07:00:00 PM,30,TRAFFIC SIGNAL,UNKNOWN,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,ONE-WAY,...,0.0,19,1,4,41.883289,-87.624549,POINT (-87.624549396399 41.883289151389),563.102174,0,99
36,b06b883b457c74cb3ba49583be68cccac67fbb76cbf1a0...,NaN,04/05/2026 06:15:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,TURNING,FOUR WAY,...,0.0,18,1,4,41.867120,-87.645281,POINT (-87.645281499953 41.86711966073),97.573644,0,92
